<a href="https://colab.research.google.com/github/Nguyennga318/Thucnghiem_NLP/blob/main/ThucnghiemNLP3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import torch
import pandas as pd
import numpy as np
import gc
import os
import time



In [ ]:
from google.colab import files

uploaded = files.upload()

Saving retrieval_results.csv to retrieval_results.csv


In [ ]:
#đọc dữ liệu
retrieval_df = pd.read_csv(
    "retrieval_results.csv",
    encoding="utf-8-sig"
)

print("Kích thước dữ liệu:", retrieval_df.shape)

display(retrieval_df.head())# hiển thị

Kích thước dữ liệu: (300, 7)


,query_id,question,method,chunk_id,rank,score,text
0,Q001,Thuật toán ML hoạt động tốt nhất khi nào?,BM25,dlbookvn_chap05.pdf_P25_C1,1,21.920890,"từng được quan sát. Nói cách khác, theo một ng..."
1,Q001,Thuật toán ML hoạt động tốt nhất khi nào?,BM25,dlbookvn_chap05.pdf_P25_C2,2,18.650490,"mà một tác nhân AI trải nghiệm, và các loại th..."
2,Q001,Thuật toán ML hoạt động tốt nhất khi nào?,BM25,dlbookvn_chap05.pdf_P16_C2,3,17.221880,"hơn. 5.2 Capacity, Overtting, Undertting Thá..."
3,Q001,Thuật toán ML hoạt động tốt nhất khi nào?,BM25,dlbookvn_chap05.pdf_P16_C1,4,17.143265,Chúng tôi sẽ thường xuyên sử dụng thuật ngữ “t...
4,Q001,Thuật toán ML hoạt động tốt nhất khi nào?,BM25,dlbookvn_chap05.pdf_P27_C3,5,14.568117,nào ta thực hiện cho thuật toán học tập nhằm g...


kiểm tra dữ liệu đầu vào

In [ ]:
required_columns = [
    "query_id",
    "question",
    "method",
    "chunk_id",
    "rank",
    "text"
]

missing_columns = [
    col for col in required_columns
    if col not in retrieval_df.columns
]

if missing_columns:
    raise ValueError(
        f"File CSV thiếu các cột: {missing_columns}"
    )

# Loại bỏ những dòng không có nội dung cần thiết
retrieval_df = retrieval_df.dropna(
    subset=[
        "query_id",
        "question",
        "method",
        "chunk_id",
        "text"
    ]
).copy()

# Chuẩn hóa kiểu dữ liệu
retrieval_df["query_id"] = (
    retrieval_df["query_id"].astype(str)
)

retrieval_df["method"] = (
    retrieval_df["method"].astype(str).str.strip()
)

retrieval_df["rank"] = pd.to_numeric(
    retrieval_df["rank"],
    errors="coerce"
)

retrieval_df["text"] = (
    retrieval_df["text"].astype(str)
)

# Sắp xếp theo thứ hạng truy xuất
retrieval_df = retrieval_df.sort_values(
    ["query_id", "method", "rank"]
)

print("Số câu hỏi:", retrieval_df["query_id"].nunique())

print("\nSố lượng kết quả theo phương pháp:")
print(retrieval_df["method"].value_counts())

print("\nSố đoạn theo từng câu hỏi và phương pháp:")
display(
    retrieval_df.groupby(
        ["query_id", "method"]
    ).size().value_counts()
)

Số câu hỏi: 20

Số lượng kết quả theo phương pháp:
method
BM25         100
Embedding    100
TF-IDF       100
Name: count, dtype: int64

Số đoạn theo từng câu hỏi và phương pháp:


,count
5,60


cài đặt thư viện, khởi tạo mô  hình

In [ ]:
!pip install -q transformers accelerate

In [ ]:

from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
#chuyển vban thành các token
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if torch.cuda.is_available():
    model_dtype = torch.float16 # giảm bộ nhớ sử dụng trên GPU so vs float32
else:
    model_dtype = torch.float32
#tải mô hình DLL để sinh văn bản
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=model_dtype,
    device_map="auto" # tự phân bổ mô hinhg trên thiết bị phù hơp
)

model.eval()

print("Đã tải mô hình:", MODEL_NAME)
print("Số chiều embedding không áp dụng cho mô hình tạo sinh.")
print("Thiết bị:", model.device)
print("Độ dài đầu vào tối đa của tokenizer:", tokenizer.model_max_length)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (2 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Đã tải mô hình: Qwen/Qwen2.5-3B-Instruct
Số chiều embedding không áp dụng cho mô hình tạo sinh.
Thiết bị: cuda:0
Độ dài đầu vào tối đa của tokenizer: 131072


xây dựng hàm tạo ngữ cảnh

In [7]:
# @title
def build_context(group, max_context_tokens=6000):

    group = group.sort_values("rank")

    context_parts = []
    used_tokens = 0

    for _, row in group.iterrows():

        chunk_id = str(row["chunk_id"])
        text = str(row["text"])

        # Đánh dấu nguồn của từng đoạn
        chunk_text = (
            f"[Nguồn: {chunk_id}]\n{text}"
        )

        tokens = tokenizer.encode(
            chunk_text,
            add_special_tokens=False
        )
        #giới hạn độ dài đầu vào
        remaining = max_context_tokens - used_tokens

        if remaining <= 0:
            break

        # Nếu đoạn cuối vượt giới hạn, cắt bớt nội dung
        if len(tokens) > remaining:
            tokens = tokens[:remaining]
            chunk_text = tokenizer.decode(
                tokens,
                skip_special_tokens=True
            )

        context_parts.append(chunk_text)

        used_tokens += len(tokens)

    context = "\n\n".join(context_parts)

    return context

xây dựng frompt tạo câu trả lời

In [11]:
def create_prompt(question, context):

    system_prompt = """
Bạn là trợ lý học thuật có nhiệm vụ trả lời câu hỏi
dựa trên tài liệu được cung cấp.

Nguyên tắc:
1. Chỉ sử dụng thông tin trong ngữ cảnh tài liệu.
2. Trả lời chính xác, rõ ràng và có tính logic.
3. Có thể tổng hợp thông tin từ nhiều đoạn văn bản.
4. Không tự ý bổ sung thông tin không có căn cứ.
5. Nếu ngữ cảnh không đủ thông tin, hãy thông báo
rằng tài liệu được truy xuất chưa cung cấp đủ thông tin.
6. Khi sử dụng thông tin từ một đoạn văn bản,
hãy ghi mã nguồn tương ứng trong ngoặc vuông,
ví dụ [chunk_12].
"""

    user_prompt = f"""
NGỮ CẢNH TÀI LIỆU:

{context}

CÂU HỎI:

{question}

Hãy đưa ra câu trả lời dựa trên ngữ cảnh trên.
"""

    messages = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    return messages

In [8]:
#hàm sinh câu trả lời
def generate_answer(question, context):

    messages = create_prompt(
        question,
        context
    )

    # Chuyển hội thoại thành định dạng đầu vào của Qwen
    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Giới hạn max_length tối ưu hơn (ví dụ 4096) để tránh quá tải bộ nhớ
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    ).to(model.device)

    # Đo thời gian sinh câu trả lời
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start_time = time.time()

    with torch.no_grad():
        # Thêm use_cache=True để tăng tốc độ sinh tự hồi quy (autoregressive generation)
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id
        )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    generation_time = time.time() - start_time

    # Chỉ giải mã phần token mới được sinh ra
    new_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    answer = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    )

    return answer.strip(), generation_time

thực nghiệm cho cả 3 phương pháp truy xuất

In [12]:
generation_results = []

# Lấy danh sách 10 query_id đầu tiên để chạy thử nghiệm kiểm chứng nhanh
test_query_ids = retrieval_df["query_id"].unique()[:10]
test_df = retrieval_df[retrieval_df["query_id"].isin(test_query_ids)]

# Nhóm dữ liệu theo câu hỏi và phương pháp truy xuất
grouped = test_df.groupby(
    ["query_id", "method"],
    sort=False
)

total_groups = len(grouped)

print(f"Tổng số lượt tạo sinh thử nghiệm (9 câu hỏi): {total_groups}")

for index, ((query_id, method), group) in enumerate(
    grouped,
    start=1
):

    group = group.sort_values("rank")

    # Lấy câu hỏi
    question = group["question"].iloc[0]

    # Lấy ID các đoạn văn bản
    retrieved_ids = group["chunk_id"].astype(str).tolist()

    # Xây dựng ngữ cảnh từ các đoạn tài liệu
    context = build_context(group)

    print("=" * 70)
    print(f"Lượt {index}/{total_groups}")
    print("Query ID:", query_id)
    print("Phương pháp:", method)
    print("Số đoạn truy xuất:", len(retrieved_ids))

    # Sinh câu trả lời
    answer, generation_time = generate_answer(
        question=question,
        context=context
    )

    print("\nCâu hỏi:", question)
    print("\nCâu trả lời:", answer)
    print("\nThời gian sinh:", round(generation_time, 3), "giây")

    # Lưu kết quả
    generation_results.append({
        "query_id": query_id,
        "question": question,
        "method": method,
        "retrieved_chunk_ids": ", ".join(retrieved_ids),
        "context": context,
        "generated_answer": answer,
        "generation_time_sec": generation_time
    })

    # Lưu tạm kết quả thực nghiệm nhanh
    pd.DataFrame(generation_results).to_csv(
        "generation_results.csv",
        index=False,
        encoding="utf-8-sig"
    )



Tổng số lượt tạo sinh thử nghiệm (9 câu hỏi): 30
Lượt 1/30
Query ID: Q001
Phương pháp: BM25
Số đoạn truy xuất: 5

Câu hỏi: Thuật toán ML hoạt động tốt nhất khi nào?

Câu trả lời: Thuật toán ML hoạt động tốt nhất khi được thiết kế để hoạt động tốt trên các phân phối sinh dữ liệu mà chúng ta quan tâm, thay vì cố gắng tìm một thuật toán học tập phổ quát hoặc tối ưu nhất trong mọi trường hợp. Mục tiêu chính là hiểu những loại phân phối nào liên quan đến "thế giới thực" mà một tác nhân AI trải nghiệm, và các loại thuật toán ML nào hoạt động tốt trên dữ liệu được lấy ra từ các loại phân phối sinh dữ liệu mà ta quan tâm.

Thời gian sinh: 6.439 giây
Lượt 2/30
Query ID: Q001
Phương pháp: Embedding
Số đoạn truy xuất: 5

Câu hỏi: Thuật toán ML hoạt động tốt nhất khi nào?

Câu trả lời: Thuật toán ML hoạt động tốt nhất khi được thiết kế để phù hợp với các phân phối sinh dữ liệu cụ thể mà chúng ta đang đối mặt trong các ứng dụng thực tế. Điều này có nghĩa là chúng ta cần hiểu rõ về loại phân phối xá

kiểm tra kết quả tạo sinh

In [13]:
generation_df = pd.DataFrame(generation_results)

print("Kích thước dữ liệu:", generation_df.shape)

print("\nSố lượt tạo sinh theo phương pháp:")
print(generation_df["method"].value_counts())

print("\nThống kê thời gian sinh:")
display(
    generation_df.groupby("method")[
        "generation_time_sec"
    ].agg(["mean", "min", "max"])
)

display(generation_df.head(5))

Kích thước dữ liệu: (30, 7)

Số lượt tạo sinh theo phương pháp:
method
BM25         10
Embedding    10
TF-IDF       10
Name: count, dtype: int64

Thống kê thời gian sinh:


,mean,min,max
method,,,
BM25,8.151973,3.693910,14.401968
Embedding,9.861078,3.810383,21.631885
TF-IDF,8.475369,3.739059,14.994885


,query_id,question,method,retrieved_chunk_ids,context,generated_answer,generation_time_sec
0,Q001,Thuật toán ML hoạt động tốt nhất khi nào?,BM25,"dlbookvn_chap05.pdf_P25_C1, dlbookvn_chap05.pd...",[Nguồn: dlbookvn_chap05.pdf_P25_C1]\ntừng được...,Thuật toán ML hoạt động tốt nhất khi được thiế...,6.438774
1,Q001,Thuật toán ML hoạt động tốt nhất khi nào?,Embedding,"dlbookvn_chap05.pdf_P25_C2, dlbookvn_chap05.pd...",[Nguồn: dlbookvn_chap05.pdf_P25_C2]\nmà một tá...,Thuật toán ML hoạt động tốt nhất khi được thiế...,21.631885
2,Q001,Thuật toán ML hoạt động tốt nhất khi nào?,TF-IDF,"dlbookvn_chap05.pdf_P25_C1, dlbookvn_chap05.pd...",[Nguồn: dlbookvn_chap05.pdf_P25_C1]\ntừng được...,Thuật toán ML hoạt động tốt nhất khi đáp ứng đ...,6.339579
3,Q002,Độ chinhs xác là gì ?,BM25,"dlbookvn_chap05.pdf_P37_C1, dlbookvn_chap05.pd...",[Nguồn: dlbookvn_chap05.pdf_P37_C1]\nPhương sa...,"Theo ngữ cảnh được cung cấp, độ chệch (bias) c...",8.084589
4,Q002,Độ chinhs xác là gì ?,Embedding,nhasachmienphi-machine-learning-co-ban.pdf_P23...,[Nguồn: nhasachmienphi-machine-learning-co-ban...,Độ chính xác trong ngữ cảnh này là tỷ lệ số đi...,12.636763


In [14]:
# kiểm tra câu tl bị trống
empty_answers = generation_df[
    generation_df["generated_answer"].isna() |
    (generation_df["generated_answer"].str.strip() == "")
]

print("Số câu trả lời trống:", len(empty_answers))

Số câu trả lời trống: 0


In [15]:
# lưu và tải file
output_file = "generation_results.csv"

generation_df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print("Đã lưu file:", output_file)


Đã lưu file: generation_results.csv


In [16]:
# tải file về máy
from google.colab import files

files.download("generation_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>